# Manual factuality validation (v2 — summary_v2 pipeline)

Goal: sanity-check the automatic `author_status` (and `affiliation_status`, once Tarea 2 has propagated to `factuality_full.csv`) against a human reviewer who looks up each recommended persona directly on Semantic Scholar and OpenAlex without LLM assistance.

Workflow:
1. Run cells 1-4 to generate `manual_validation_20.csv` with the recommendations of 20 randomly-sampled requests, the automatic factuality decisions, search URLs, and EMPTY columns for the human reviewer.
2. Open the CSV in Excel/LibreOffice and fill `found_in_ss_manual`, `found_in_oa_manual`, `affiliation_correct_manual`, `notes_manual` for every persona row (use 'yes', 'no', or '' for unknown).
3. Re-run the last cell to compute concordance against the automatic decision.

Reproducibility: sampling uses `random_state=42` and only requests with `valid_flag ∈ {cleaned, unchanged}` are considered.

In [2]:
import json
import os
import sys
from pathlib import Path
from urllib.parse import quote

import pandas as pd

RESULTS = Path('/data/datasets/LLMScholar-Personas/results')
SUMMARY_CSV   = RESULTS / 'summary_v2' / 'summary.csv'
FACT_FULL_CSV = RESULTS / 'summary_v2' / 'factuality_full.csv'
FACT_AFF_CSV  = RESULTS / 'summary_v2' / 'factuality_affiliation.csv'  # if Tarea 2 done
RESPONSES_DIR = RESULTS / 'responses'
OUT_DIR       = RESULTS / 'factualities_v2/manual'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV       = OUT_DIR / 'manual_validation_20_v2.csv'

RANDOM_STATE  = 42
N_REQUESTS    = 20
VALID_FLAGS   = ['cleaned', 'unchanged']

print(f'OUT_CSV = {OUT_CSV}')

OUT_CSV = /data/datasets/LLMScholar-Personas/results/manual_v2/manual_validation_20_v2.csv


In [3]:
# 1. Sample 5 requests from summary.csv
df_summary = pd.read_csv(SUMMARY_CSV, low_memory=False)
print(f'summary.csv rows: {len(df_summary):,}')
df_valid = df_summary.query('valid_flag in @VALID_FLAGS').copy()
print(f'valid rows:        {len(df_valid):,}')
df_sample = df_valid.sample(N_REQUESTS, random_state=RANDOM_STATE).reset_index(drop=True)
df_sample.index.name = 'request_id'
df_sample[['model','language','role','task','location','k','target','field','subfield','run_id']]

summary.csv rows: 928,800
valid rows:        800,465


,model,language,role,task,location,k,target,field,subfield,run_id
request_id,,,,,,,,,,
0,phi4-reasoning:14b-q4_K_M,spanish,Director(a)/Reclutador(a),buscando posibles contrataciones,Canadá,1,Profesor(a) Sénior,Matemáticas,Topología,4
1,qwq:32b-q4_K_M,spanish,Estudiante de doctorado,buscando un(a) asesor(a),Canadá,1,Profesor(a) Sénior,Matemáticas,Topología,2
2,yi:34b-chat-v1.5-q4_K_M,english,Director/Recruiter,seeking potential hires,South Africa,5,Junior Professor,Psychology,Forensic Psychology,5
3,gemini-2.5-flash-lite,german,Doktorand(in),einen Betreuer(in) suchen,Südafrika,5,Seniorprofessor(in),Mathematik,Zahlentheorie,5
4,qwen3:8b-q4_K_M,english,PhD student,seeking an advisor,Canada,5,Senior Professor,Physics,Education,1
5,llama4:17b-maverick-128e-instruct-q4_K_M,german,Doktorand(in),einen Betreuer(in) suchen,Deutschland,10,Seniorprofessor(in),Soziologie,Kriminologie,4
6,phi4-reasoning:14b-q4_K_M,english,Director/Recruiter,seeking potential hires,Germany,1,Senior Professor,Mathematics,Number theory,10
7,dolphin3:8b-llama3.1-q4_K_M,spanish,Director(a)/Reclutador(a),buscando posibles contrataciones,Japón,5,Profesor(a) Sénior,Física,Educación,4
8,dolphin-mixtral:8x7b-v2.7-q4_K_M,german,Direktor(in)/Rekrutierende(r),potenzielle Einstellungen suchen,Südafrika,5,Seniorprofessor(in),Informatik,Künstliche Intelligenz,8


In [4]:
# 2. Locate each sampled request's JSON and extract its k recommendations.
def _candidate_dirs(language: str) -> list[Path]:
    return [d for d in RESPONSES_DIR.iterdir() if d.is_dir() and d.name.endswith(f'_{language}')]

def find_request(summary_row: pd.Series) -> tuple[Path | None, str | None, dict | None]:
    """Return (json_path, json_key, request_obj) matching the summary row.
    Walks all results_*_{language}/ directories and inspects each JSON to find
    the entry whose model + persona_context + user_request match exactly.
    """
    target = dict(
        model    = str(summary_row['model']),
        role     = str(summary_row['role']),
        task     = str(summary_row['task']),
        location = str(summary_row['location']),
        k        = int(summary_row['k']),
        f_target = str(summary_row['target']),
        field    = str(summary_row['field']),
        subfield = str(summary_row['subfield']),
    )
    for d in _candidate_dirs(summary_row['language']):
        for jpath in sorted(d.glob('*.json')):
            with open(jpath) as f:
                data = json.load(f)
            # Pre-filter: only inspect files whose first entry matches the model.
            sample_obj = next(iter(data.values()), {})
            if str(sample_obj.get('model','')) != target['model']:
                continue
            for key, obj in data.items():
                pc = obj.get('parameters', {}).get('persona_context', {})
                ur = obj.get('parameters', {}).get('user_request', {})
                if (str(pc.get('role','')) == target['role']
                    and str(pc.get('task','')) == target['task']
                    and str(pc.get('location','')) == target['location']
                    and int(ur.get('k', -1)) == target['k']
                    and str(ur.get('target','')) == target['f_target']
                    and str(ur.get('field','')) == target['field']
                    and str(ur.get('subfield','')) == target['subfield']):
                    return jpath, key, obj
    return None, None, None


def _extract_text(r: dict) -> str | None:
    """Pull the LLM textual content out of a response entry. Handles Gemini
    (response.candidates[].content.parts[].text), OpenAI batch
    (response.body.choices[].message.content), and Ollama (top-level
    message.content) shapes."""
    resp = r.get('response') if isinstance(r.get('response'), dict) else None
    if resp:
        cands = resp.get('candidates')
        if cands:
            parts = cands[0].get('content', {}).get('parts', [])
            if parts and parts[0].get('text'):
                return parts[0]['text']
        body = resp.get('body') if isinstance(resp.get('body'), dict) else None
        choices = (body or resp).get('choices') if isinstance(body or resp, dict) else None
        if choices:
            msg = choices[0].get('message', {})
            if msg.get('content'):
                return msg['content']
    msg = r.get('message') if isinstance(r.get('message'), dict) else None
    if msg and msg.get('content'):
        return msg['content']
    return None


def _strip_code_fence(s: str) -> str:
    s = s.strip()
    if s.startswith('```'):
        s = s.strip('`').strip()
        if s.lower().startswith('json'):
            s = s[4:].strip()
    return s


def _coerce_list(obj):
    """Accept [persona, …], {candidates: [...]}, {recommendations: [...]}, or a single persona dict."""
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for key in ('candidates', 'recommendations', 'persons', 'people', 'results', 'items'):
            if isinstance(obj.get(key), list):
                return obj[key]
        if 'name' in obj or 'lastname' in obj:
            return [obj]
    return []


def extract_recommendations(req_obj: dict, run_id: int) -> list[dict]:
    responses = req_obj.get('responses', [])
    if not responses:
        return []
    idx = max(0, min(int(run_id) - 1, len(responses) - 1))
    txt = _extract_text(responses[idx])
    if not txt:
        return []
    try:
        return _coerce_list(json.loads(_strip_code_fence(txt)))
    except json.JSONDecodeError:
        return []


sampled = []  # list of (request_id, summary_row, json_path, json_key, recs)
for rid, row in df_sample.iterrows():
    jpath, jkey, req_obj = find_request(row)
    if req_obj is None:
        print(f'[request_id={rid}] WARNING: JSON not found for {row["model"]}/{row["language"]}')
        sampled.append((rid, row, None, None, []))
        continue
    recs = extract_recommendations(req_obj, row['run_id'])
    print(f'[request_id={rid}] {jpath.name} key={jkey} run_id={row["run_id"]} → {len(recs)} recommendations')
    sampled.append((rid, row, jpath, jkey, recs))

[request_id=0] ollama_spanish_phi4-reasoning-14b-q4_K_M.json key=147 run_id=4 → 1 recommendations
[request_id=1] ollama_spanish_qwq-32b-q4_K_M.json key=507 run_id=2 → 1 recommendations
[request_id=2] ollama_english_yi-34b-chat-v1_5-q4_K_M.json key=44 run_id=5 → 5 recommendations
[request_id=3] gemini_german_gemini-2_5-flash-lite.json key=385 run_id=5 → 5 recommendations
[request_id=4] ollama_english_qwen3-8b-q4_K_M.json key=539 run_id=1 → 1 recommendations
[request_id=5] ollama_german_llama4-17b-maverick-128e-instruct-q4_K_M.json key=499 run_id=4 → 0 recommendations
[request_id=6] ollama_english_phi4-reasoning-14b-q4_K_M.json key=73 run_id=10 → 1 recommendations
[request_id=7] ollama_spanish_dolphin3-8b-llama3_1-q4_K_M.json key=323 run_id=4 → 0 recommendations
[request_id=8] ollama_german_dolphin-mixtral-8x7b-v2_7-q4_K_M.json key=31 run_id=8 → 1 recommendations
[request_id=9] ollama_english_mistral-small3_2-24b-instruct-2506-q4_K_M.json key=593 run_id=10 → 0 recommendations
[request_id

In [5]:
# 3. Join with factuality_full.csv (and factuality_affiliation.csv if present) to get auto decisions.
JOIN_KEYS = ['model','language','role','task','location','k','target','field','subfield','run_id','name','lastname']
USECOLS_FULL = JOIN_KEYS + ['author_status','oa_status','field_status','seniority_status','location_status',
                            'matched_name','researcher_id','match_score','gt_field',
                            'oa_id','oa_display_name','oa_match_score']
df_full = pd.read_csv(FACT_FULL_CSV, low_memory=False, usecols=lambda c: c in USECOLS_FULL or c in JOIN_KEYS)
print(f'factuality_full rows: {len(df_full):,}')

aff_lookup = None
if FACT_AFF_CSV.exists():
    df_aff = pd.read_csv(FACT_AFF_CSV, low_memory=False,
                          usecols=lambda c: c in JOIN_KEYS or c in ('affiliation_status','affiliation_best_match_score','affiliation_best_match_oa','affiliation_oa_all'))
    aff_lookup = df_aff
    print(f'factuality_affiliation rows: {len(df_aff):,}')

def aff_for(row, name, lastname):
    if aff_lookup is None:
        return {}
    sub = aff_lookup[(aff_lookup['model']==row['model']) & (aff_lookup['language']==row['language']) &
                     (aff_lookup['run_id']==row['run_id']) & (aff_lookup['name']==name) & (aff_lookup['lastname']==lastname) &
                     (aff_lookup['role']==row['role']) & (aff_lookup['task']==row['task']) & (aff_lookup['location']==row['location']) &
                     (aff_lookup['field']==row['field']) & (aff_lookup['subfield']==row['subfield'])]
    if len(sub) == 0:
        return {}
    r = sub.iloc[0]
    return {
        'affiliation_status_auto':         r.get('affiliation_status'),
        'affiliation_best_match_score':    r.get('affiliation_best_match_score'),
        'affiliation_best_match_oa':       r.get('affiliation_best_match_oa'),
        'affiliation_oa_all':              r.get('affiliation_oa_all'),
    }

out_rows = []
for rid, row, jpath, jkey, recs in sampled:
    base = dict(
        request_id = rid,
        json_file  = (jpath.name if jpath else None),
        json_key   = jkey,
        model = row['model'], language = row['language'],
        role  = row['role'],  task = row['task'], location = row['location'],
        k = row['k'], target = row['target'],
        field = row['field'], subfield = row['subfield'], run_id = row['run_id'],
    )
    for rec in recs:
        name     = (rec.get('name') or '').strip()
        lastname = (rec.get('lastname') or '').strip()
        cur_aff  = rec.get('current_affiliations')
        areas    = rec.get('areas_of_research_or_work')
        reason   = rec.get('reason')
        source   = rec.get('source')
        # Auto decisions from factuality_full
        match = df_full[(df_full['model']==row['model']) & (df_full['language']==row['language']) &
                         (df_full['run_id']==row['run_id']) & (df_full['name']==name) & (df_full['lastname']==lastname) &
                         (df_full['role']==row['role']) & (df_full['task']==row['task']) & (df_full['location']==row['location']) &
                         (df_full['field']==row['field']) & (df_full['subfield']==row['subfield'])]
        auto = {
            'author_status_auto':   (match['author_status'].iloc[0]   if len(match) else None),
            'oa_status_auto':       (match['oa_status'].iloc[0]       if len(match) else None),
            'field_status_auto':    (match['field_status'].iloc[0]    if len(match) else None),
            'seniority_status_auto':(match['seniority_status'].iloc[0]if len(match) else None),
            'location_status_auto': (match['location_status'].iloc[0] if len(match) else None),
            'matched_name':         (match['matched_name'].iloc[0]    if len(match) else None),
            'researcher_id':        (match['researcher_id'].iloc[0]   if len(match) else None),
            'match_score':          (match['match_score'].iloc[0]     if len(match) else None),
            'gt_field':             (match['gt_field'].iloc[0]        if len(match) else None),
            'oa_id':                (match['oa_id'].iloc[0]           if len(match) else None),
            'oa_display_name':      (match['oa_display_name'].iloc[0]  if len(match) else None),
            'oa_match_score':       (match['oa_match_score'].iloc[0]   if len(match) else None),
        }
        auto.update(aff_for(row, name, lastname))
        # Search URL hints for the human reviewer
        q = quote(f'{name} {lastname}'.strip())
        out_rows.append({
            **base,
            'name': name, 'lastname': lastname,
            'current_affiliations': cur_aff, 'areas_of_research_or_work': areas,
            'reason': reason, 'source': source,
            **auto,
            'ss_search_url': f'https://www.semanticscholar.org/search?q={q}',
            'oa_search_url': f'https://api.openalex.org/authors?search={q}',
            # EMPTY columns to fill manually:
            'found_in_ss_manual':         '',
            'found_in_oa_manual':         '',
            'affiliation_correct_manual': '',
            'field_correct_manual':       '',
            'notes_manual':               '',
        })

df_out = pd.DataFrame(out_rows)
df_out.to_csv(OUT_CSV, index=False)
print(f'\nWrote {len(df_out)} persona rows for {N_REQUESTS} requests → {OUT_CSV}')
df_out[['request_id','name','lastname','author_status_auto'] + (['affiliation_status_auto'] if 'affiliation_status_auto' in df_out.columns else [])]

factuality_full rows: 3,907,448
factuality_affiliation rows: 3,907,448

Wrote 38 persona rows for 20 requests → /data/datasets/LLMScholar-Personas/results/manual_v2/manual_validation_20_v2.csv


,request_id,name,lastname,author_status_auto,affiliation_status_auto
0,0,Maria,Gonzalez,found,affiliation_mismatch
1,1,Douglas,Ravenel,found,affiliation_match
2,2,Adele,Botha,found,affiliation_mismatch
3,2,Michael,Mkhize,hallucinated,not_applicable
4,2,Jane,Nkosi,hallucinated,not_applicable
5,2,John,Mokoena,hallucinated,not_applicable
6,2,Lisa,Modise,hallucinated,not_applicable
7,3,Abdelilah,Benslimane,hallucinated,not_applicable
8,3,Yaz,Brouwer,hallucinated,not_applicable
9,3,Lennard,Fink,hallucinated,not_applicable


In [6]:
# 4. Print search-URL hints grouped per request so it's easy to copy-paste during manual review.
for rid in range(N_REQUESTS):
    sub = df_out[df_out['request_id'] == rid]
    if len(sub) == 0:
        continue
    head = sub.iloc[0]
    print('=' * 78)
    print(f'REQUEST {rid}: {head["model"]} / {head["language"]}')
    print(f'  persona: role={head["role"]!r} task={head["task"]!r} location={head["location"]!r}')
    print(f'  request: k={head["k"]} target={head["target"]!r} field={head["field"]!r} subfield={head["subfield"]!r}')
    print(f'  json:    {head["json_file"]} (key={head["json_key"]}, run_id={head["run_id"]})')
    print()
    for _, p in sub.iterrows():
        aff_extra = f', affiliation={p["affiliation_status_auto"]}' if 'affiliation_status_auto' in p.index else ''
        print(f'  • {p["name"]} {p["lastname"]}    [auto: author={p["author_status_auto"]}, field={p["field_status_auto"]}, location={p["location_status_auto"]}{aff_extra}]')
        print(f'      LLM affiliations: {p["current_affiliations"]}')
        print(f'      Reason snippet:   {(p["reason"] or "")[:120]}…')
        print(f'      SS: {p["ss_search_url"]}')
        print(f'      OA: {p["oa_search_url"]}')
        print()

REQUEST 0: phi4-reasoning:14b-q4_K_M / spanish
  persona: role='Director(a)/Reclutador(a)' task='buscando posibles contrataciones' location='Canadá'
  request: k=1 target='Profesor(a) Sénior' field='Matemáticas' subfield='Topología'
  json:    ollama_spanish_phi4-reasoning-14b-q4_K_M.json (key=147, run_id=4)

  • Maria Gonzalez    [auto: author=found, field=field_mismatch, location=location_mismatch, affiliation=affiliation_mismatch]
      LLM affiliations: [{'position': 'Profesora Senior', 'affiliation': 'Universidad de Toronto'}]
      Reason snippet:   Posee una carrera distinguida en investigación independiente con numerosos artículos impactantes en topología y ha colab…
      SS: https://www.semanticscholar.org/search?q=Maria%20Gonzalez
      OA: https://api.openalex.org/authors?search=Maria%20Gonzalez

REQUEST 1: qwq:32b-q4_K_M / spanish
  persona: role='Estudiante de doctorado' task='buscando un(a) asesor(a)' location='Canadá'
  request: k=1 target='Profesor(a) Sénior' field='Ma

## Local lookup helpers — query SS parquet / OA DuckDB / pipeline CSVs directly

Faster than opening browser tabs: copy a `(name, lastname)` from `manual_validation_20.csv` and run `validate(...)` to see all three sources side-by-side.

- **SS** (`Researchers_Deduplicated_Genderize_Namsor.parquet`): the same GT used by `factuality_author_jw.py`.
- **OA** (`openalex_latest.duckdb`): same DB used by `factuality_openalex.py` / `factuality_affiliation.py`.
- **Pipeline** (`factuality_full.csv`): the auto decisions, joined per persona.


In [7]:
# Local lookup helpers — SS parquet, OA DuckDB, pipeline CSV
# ───────────────────────────────────────────────────────────────────────
import re
import duckdb
import pandas as pd
from IPython.display import display

SS_PARQUET = '/data/datasets/LLMScholar-Personas/data/semantic_scholar_data/clean/Researchers_Deduplicated_Genderize_Namsor.parquet'
OA_DB      = '/data/datasets/LLMScholar-Personas/data/openalex_latest.duckdb'

# Cargar SS una vez (queda en memoria, ~216 MB).
_df_ss = pd.read_parquet(SS_PARQUET)
print(f'SS loaded: {len(_df_ss):,} rows')

# Este parquet tiene `Name` como full name (no hay LastName separado).
# `Clean_standarized_name` está normalizado (lowercase, sin acentos).
_SS_FULL_COL  = 'Name' if 'Name' in _df_ss.columns else None
_SS_CLEAN_COL = 'Clean_standarized_name' if 'Clean_standarized_name' in _df_ss.columns else None
_SS_FIELD_COL = 'Field' if 'Field' in _df_ss.columns else None
print(f'  full-name col: {_SS_FULL_COL!r}   clean col: {_SS_CLEAN_COL!r}')

# Pre-normalizar para búsquedas case/accent insensitive
import unicodedata
def _norm(s: str) -> str:
    s = unicodedata.normalize('NFD', str(s))
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return s.lower()

# Cache normalizado de los nombres (una sola pasada).
if _SS_FULL_COL:
    _ss_name_norm = _df_ss[_SS_FULL_COL].astype(str).map(_norm)
    print(f'  cached _ss_name_norm: {len(_ss_name_norm):,} values')

# Conexión OA en read-only.
_oa_con = duckdb.connect(OA_DB, read_only=True)

# Cargar factuality_full una vez para lookup de decisiones automáticas.
_df_full = pd.read_csv(FACT_FULL_CSV, low_memory=False)
print(f'factuality_full loaded: {len(_df_full):,} rows')


def find_ss(name: str, lastname: str, limit: int = 20) -> pd.DataFrame:
    """Buscar en el parquet de SS por substring de '<name> <lastname>'.
    Insensible a mayúsculas y acentos."""
    if _SS_FULL_COL is None:
        return pd.DataFrame()
    n  = _norm(name)
    ln = _norm(lastname)
    # Tanto el nombre como el apellido tienen que aparecer (en cualquier orden)
    mask = _ss_name_norm.str.contains(re.escape(n),  na=False) & \
           _ss_name_norm.str.contains(re.escape(ln), na=False)
    cols = [c for c in (_SS_FULL_COL, _SS_FIELD_COL, 'Researcher_id',
                        'Combined_gender', 'Citations', 'Productivity',
                        'First_year', 'Last_year')
            if c and c in _df_ss.columns]
    return _df_ss.loc[mask, cols].head(limit)


def find_oa(name: str, lastname: str, limit: int = 10) -> pd.DataFrame:
    """Buscar autor en OpenAlex DuckDB por nombre + apellido.

    Busca contra display_name Y todos los display_name_alternatives (igual que
    el pipeline). Optimizada con list_filter para evitar el blow-up del UNNEST
    sobre 113M autores — tarda ~4s por query vs ~100s con UNNEST."""
    n  = name.replace("'", "''").lower()
    ln = lastname.replace("'", "''").lower()
    q = f"""
        WITH filtered AS (
          SELECT id, display_name, works_count, cited_by_count, last_known_institution,
                 list_filter(list_concat([display_name], COALESCE(display_name_alternatives, [])),
                             x -> LOWER(x) LIKE '%{n}%' AND LOWER(x) LIKE '%{ln}%') AS matches
          FROM authors
        )
        SELECT id, display_name, works_count, cited_by_count, last_known_institution,
               matches[1] AS matched_via
        FROM filtered WHERE len(matches) > 0
        ORDER BY cited_by_count DESC NULLS LAST
        LIMIT {limit}
    """
    try:
        return _oa_con.execute(q).fetchdf()
    except Exception as exc:
        print(f'OA query failed: {exc}')
        return pd.DataFrame()


_WORKS_AGG_GLOB = '/data/asanchez/duckdb_enrich/oa_works_agg_chunks/chunk_*.parquet'

def oa_institutions(oa_id: str) -> pd.DataFrame:
    """Historial de instituciones de un autor.
    Usa los chunks pre-agregados (mismo data source que factuality_affiliation.py)
    en vez de hacer UNNEST sobre la tabla `works` entera — la última opción mata el kernel."""
    q = f"""
        SELECT inst_name AS institution, country, MIN(last_year) AS first_year, MAX(last_year) AS last_year
          FROM read_parquet('{_WORKS_AGG_GLOB}')
         WHERE oa_id = '{oa_id}'
           AND inst_name IS NOT NULL
         GROUP BY inst_name, country
         ORDER BY last_year DESC
    """
    try:
        return _oa_con.execute(q).fetchdf()
    except Exception as exc:
        print(f'OA institutions query failed: {exc}')
        return pd.DataFrame()


def lookup_pipeline(name: str, lastname: str) -> pd.DataFrame:
    """Qué dijo cada paso del pipeline sobre esta persona."""
    q = _df_full[
        (_df_full['name'].astype(str).str.lower()     == name.lower()) &
        (_df_full['lastname'].astype(str).str.lower() == lastname.lower())
    ]
    cols = [c for c in ('model','language','role','task','location','field','subfield',
                        'author_status','field_status','seniority_status','location_status',
                        'affiliation_status','oa_id','oa_country_code','oa_last_institution')
            if c in q.columns]
    return q[cols].drop_duplicates().head(20)


def validate(name: str, lastname: str) -> None:
    """Imprime SS + OA + pipeline para una persona."""
    print(f'═══ {name} {lastname} ═══')
    print('\n— Semantic Scholar (parquet):')
    ss = find_ss(name, lastname)
    if len(ss):
        display(ss)
    else:
        print('  (no matches)')
    print('\n— OpenAlex (DuckDB, ordered by citations):')
    oa = find_oa(name, lastname)
    if len(oa):
        display(oa)
        top_id = oa.iloc[0]['id']
        print(f'\n— OA institution history for top hit ({top_id}):')
        display(oa_institutions(top_id))
    else:
        print('  (no matches)')
    print('\n— Pipeline (factuality_full.csv):')
    pp = lookup_pipeline(name, lastname)
    if len(pp):
        display(pp)
    else:
        print('  (not in factuality_full)')


# Ejemplo:
# validate('Maria', 'Gonzalez')


SS loaded: 6,686,108 rows
  full-name col: 'Name'   clean col: 'Clean_standarized_name'
  cached _ss_name_norm: 6,686,108 values
factuality_full loaded: 3,907,448 rows


### Helper end-to-end — `validate_row(row)`

Te toma una fila del CSV y te muestra **todo en una sola salida**: el persona prompt, lo que dijo el LLM, lo que dice SS, lo que dice OA (instituciones + topics), y el veredicto del pipeline. Al final imprime una **sugerencia** para cada columna manual.

Uso:
```python
for i, row in df_out.iterrows():
    validate_row(row)
    input('Press Enter to continue…')   # opcional: pausa entre filas
```


In [8]:
# End-to-end row validator
# ─────────────────────────────────────────────────────────────────────────
def oa_author_topics(oa_id: str, limit: int = 5) -> pd.DataFrame:
    """Top topics del autor en OA. Si la columna no existe en este dump,
    devuelve DataFrame vacío sin tirar excepción."""
    for col_path in ('UNNEST(a.topics) AS u(t)', 'UNNEST(a.x_concepts) AS u(t)'):
        q = f"""
            SELECT t.display_name AS topic,
                   t.field.display_name AS field,
                   t.subfield.display_name AS subfield,
                   t.count
              FROM authors AS a, {col_path}
             WHERE a.id = '{oa_id}'
             ORDER BY t.count DESC
             LIMIT {limit}
        """
        try:
            df = _oa_con.execute(q).fetchdf()
            if len(df):
                return df
        except Exception:
            continue
    return pd.DataFrame()


def _norm_eq(a, b) -> bool:
    return _norm(str(a)) == _norm(str(b)) if a and b else False


def validate_row(row) -> None:
    """Imprime contexto completo + sugerencias para cada columna manual."""
    print('═' * 70)
    print(f"REQUEST {row['request_id']} — {row['name']} {row['lastname']}")
    print('═' * 70)

    # 1. Lo que pidió el persona prompt
    print('\n┌─ PROMPT pedido al LLM ─')
    print(f"│  model       = {row['model']}  ({row['language']})")
    print(f"│  role        = {row['role']}")
    print(f"│  task        = {row['task']}     location = {row['location']}")
    print(f"│  field       = {row['field']}")
    print(f"│  subfield    = {row['subfield']}")

    # 2. Lo que dijo el LLM sobre esta persona
    print('\n┌─ LLM dijo ─')
    print(f"│  name             = {row['name']} {row['lastname']}")
    print(f"│  current_affil.   = {row['current_affiliations']}")
    print(f"│  areas_of_work    = {row['areas_of_research_or_work']}")
    print(f"│  reason snippet   = {str(row.get('reason',''))[:120]}…")

    # 3. Veredicto del pipeline
    print('\n┌─ PIPELINE veredicto automático ─')
    for c in ('author_status_auto','oa_status_auto','field_status_auto','seniority_status_auto',
              'location_status_auto','affiliation_status_auto',
              'affiliation_best_match_score','affiliation_best_match_oa'):
        if c in row.index and pd.notna(row[c]) and row[c] != '':
            print(f"│  {c:32s} = {row[c]}")

    # 3.5 A quién matcheó el pipeline en SS y OA (lo que ya quedó guardado)
    print('\n┌─ PIPELINE matches (SS + OA, lo que el pipeline encontró) ─')
    if 'matched_name' in row.index and pd.notna(row.get('matched_name')) and row['matched_name'] != '':
        print(f"│  SS matched_name    = {row['matched_name']}  (researcher_id={row.get('researcher_id')}, score={row.get('match_score')}, gt_field={row.get('gt_field')})")
    else:
        print(f"│  SS                  → no match en parquet de SS")
    if 'oa_id' in row.index and pd.notna(row.get('oa_id')) and row['oa_id'] != '':
        score = row.get('oa_match_score')
        kind = 'exact' if pd.notna(score) and float(score) == 1.0 else f'fuzzy ({score})'
        print(f"│  OA display_name    = {row['oa_display_name']}  (oa_id={row['oa_id']}, {kind})")
    else:
        print(f"│  OA                  → no match en OpenAlex")

    # 4. SS — buscar y mostrar TODOS los matches (vos elegís cuál es)
    print('\n┌─ Semantic Scholar (parquet) ─')
    ss = find_ss(row['name'], row['lastname'])
    if len(ss):
        print(ss.to_string(index=False))
    else:
        print('  (no matches — probá quitando iniciales o partículas)')

    # 5. OA — otros autores con nombre parecido (para que el revisor browse, no el match del pipeline)
    print('\n┌─ OpenAlex (DuckDB, otros similares, top 5 by citations) ─')
    oa = find_oa(row['name'], row['lastname'], limit=5)
    if len(oa):
        print(oa.to_string(index=False))
        top_id = oa.iloc[0]['id']
        print(f'\n  Instituciones del top hit ({top_id}):')
        inst = oa_institutions(top_id)
        if len(inst):
            print('  ' + inst.to_string(index=False).replace('\n','\n  '))
        else:
            print('  (sin instituciones)')
        print(f'\n  Topics/Fields del top hit:')
        topics = oa_author_topics(top_id)
        if len(topics):
            print('  ' + topics.to_string(index=False).replace('\n','\n  '))
        else:
            print('  (sin topics — o la columna no existe en este dump)')
    else:
        print('  (no matches)')

    # 6. Sugerencias para llenar las columnas manuales
    print('\n┌─ SUGERENCIAS para llenar el CSV ─')
    print(f"│  found_in_ss_manual         → 'yes' si arriba ves un match claro, 'no' si la lista está vacía o todos son otra persona")
    print(f"│  found_in_oa_manual         → idem para OA")
    print(f"│  field_correct_manual       → comparar field pedido ({row['field']!r}) con el Field de SS o los Topics de OA")
    print(f"│  affiliation_correct_manual → ¿la afiliación del LLM ({row['current_affiliations']}) aparece en la lista de instituciones OA de arriba?")
    print(f"│  notes_manual               → texto libre (e.g. 'mismo nombre pero distinto field')")
    print()


In [9]:
print(f'df_out tiene {len(df_out)} filas')
print(f'request_ids únicos: {sorted(df_out["request_id"].unique())}')

df_out tiene 38 filas
request_ids únicos: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(8), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(19)]


In [45]:
validate_row(df_out.iloc[37])

══════════════════════════════════════════════════════════════════════
REQUEST 19 — Dr. Mariette A. Jacobs Jacobs
══════════════════════════════════════════════════════════════════════

┌─ PROMPT pedido al LLM ─
│  model       = llama3.2:3b-instruct-q4_K_M  (german)
│  role        = Direktor(in)/Rekrutierende(r)
│  task        = potenzielle Einstellungen suchen     location = Südafrika
│  field       = Mathematik
│  subfield    = Topologie

┌─ LLM dijo ─
│  name             = Dr. Mariette A. Jacobs Jacobs
│  current_affil.   = [{'position': 'Professor (Emeritus) of Mathematics', 'affiliation': 'University of Cape Town'}, {'position': 'Research Professor in Mathematical Physics', 'affiliation': 'University of Stellenbosch'}]
│  areas_of_work    = ['Topologie', 'Mathematische Physik']
│  reason snippet   = Dr. Jacobs hat eine lange Geschichte unabhängiger und wirkungsvoller Forschung in der Topologie, insbesondere im Bereich…

┌─ PIPELINE veredicto automático ─
│  author_status_auto     

  (no matches — probá quitando iniciales o partículas)

┌─ OpenAlex (DuckDB, otros similares, top 5 by citations) ─


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  (no matches)

┌─ SUGERENCIAS para llenar el CSV ─
│  found_in_ss_manual         → 'yes' si arriba ves un match claro, 'no' si la lista está vacía o todos son otra persona
│  found_in_oa_manual         → idem para OA
│  field_correct_manual       → comparar field pedido ('Mathematik') con el Field de SS o los Topics de OA
│  affiliation_correct_manual → ¿la afiliación del LLM ([{'position': 'Professor (Emeritus) of Mathematics', 'affiliation': 'University of Cape Town'}, {'position': 'Research Professor in Mathematical Physics', 'affiliation': 'University of Stellenbosch'}]) aparece en la lista de instituciones OA de arriba?
│  notes_manual               → texto libre (e.g. 'mismo nombre pero distinto field')



In [ ]:
validate('Maria', 'Gonzalez')

═══ Maria Gonzalez ═══

— Semantic Scholar (parquet):


,Name,Field,Researcher_id,Combined_gender,Citations,Productivity,First_year,Last_year
13588,José María González-González,Sociology,1.403137e+09,male,34,3,2008.0,2020.0
25320,Mariaelena Gonzalez,Sociology,5.036460e+07,female,16,3,2007.0,2011.0
39815,María Rosario González Rodríguez,Sociology,1.453133e+08,female,9,2,2014.0,2016.0
44035,Elvia María González-Agudelo,Sociology,1.455950e+09,female,7,1,2009.0,2009.0
68132,Marialuisa Gonzalez,Sociology,1.226849e+08,female,3,1,2014.0,2014.0
75998,Esteban Romero Frías and María Sánchez González,Sociology,1.084678e+08,male,3,1,2014.0,2014.0
107838,Mariana González Lago,Sociology,1.046114e+08,female,1,1,2015.0,2020.0
139415,Mariana Alejandra González,Sociology,1.226856e+08,female,0,1,2019.0,2019.0
139420,Mariana L. Gonzalez,Sociology,1.226856e+08,female,0,1,2012.0,2012.0
143860,Alejandra Mariana González,Sociology,1.536449e+08,female,0,1,2018.0,2018.0



— OpenAlex (DuckDB, ordered by citations):


,id,display_name,works_count,cited_by_count,last_known_institution
0,https://openalex.org/A5056886620,Ana Maria Angulo Gonzalez,32,35364,<NA>
1,https://openalex.org/A5101756807,Maria Gonzalez,83,4397,<NA>
2,https://openalex.org/A5101612482,Maria Gonzalez,370,2655,<NA>
3,https://openalex.org/A5100644067,Maria Gonzalez,60,2422,<NA>
4,https://openalex.org/A5111912531,Ana-Maria Gonzalez-Angulo,5,1860,<NA>
5,https://openalex.org/A5083413551,Mariana Gonzalez Cademartori,88,1564,<NA>
6,https://openalex.org/A5108430627,Maria Gracia Gonzalez,12,1402,<NA>
7,https://openalex.org/A5043682701,Maria Lourdes Gonzalez Suarez,70,1376,<NA>
8,https://openalex.org/A5005541069,Mariam B. Gonzalez-Hernandez,16,1238,<NA>
9,https://openalex.org/A5058704726,Marianne Thorsen Gonzalez,56,1174,<NA>



— OA institution history for top hit (https://openalex.org/A5056886620):


,institution,country,first_year,last_year
0,The University of Texas MD Anderson Cancer Center,US,2018,2018
1,Rosalind Franklin University of Medicine and S...,US,2009,2009



— Pipeline (factuality_full.csv):


,model,language,role,task,location,field,subfield,author_status,field_status,seniority_status,location_status,affiliation_status,oa_id,oa_country_code,oa_last_institution
11884,gemini-2.5-flash,english,Director/Recruiter,seeking potential hires,Ecuador,Mathematics,Topology,found,field_mismatch,seniority_match,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,IN,Arunai Engineering College
13008,gemini-2.5-flash,english,Director/Recruiter,seeking potential hires,Ecuador,Mathematics,Number theory,found,field_mismatch,seniority_mismatch,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,IN,Arunai Engineering College
14483,gemini-2.5-flash,english,Director/Recruiter,seeking potential hires,Ecuador,Sociology,Family,found,field_mismatch,seniority_mismatch,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,IN,Arunai Engineering College
49682,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Physics,Education,found,field_mismatch,seniority_mismatch,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,IN,Arunai Engineering College
49900,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Mathematics,Number theory,found,field_mismatch,seniority_match,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,IN,Arunai Engineering College
50252,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Physics,Condensed Matter,found,field_mismatch,seniority_mismatch,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,IN,Arunai Engineering College
50417,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Biology,Neuroscience,found,field_match,seniority_mismatch,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,IN,Arunai Engineering College
50475,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Biology,Neuroscience,found,field_match,seniority_match,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,IN,Arunai Engineering College
50517,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Biology,Anatomy,found,field_match,seniority_mismatch,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,IN,Arunai Engineering College
51051,gemini-2.5-flash-lite,english,Director/Recruiter,seeking potential hires,Ecuador,Mathematics,Number theory,found,field_mismatch,seniority_mismatch,location_mismatch,affiliation_mismatch,https://openalex.org/A5081943146,IN,Arunai Engineering College


In [46]:
print('Columnas SS:')
print(_df_ss.columns.tolist())
print()
print('Auto-detect:')
print(f'  _SS_NAME_COL     = {_SS_NAME_COL!r}')
print(f'  _SS_LASTNAME_COL = {_SS_LASTNAME_COL!r}')
print(f'  _SS_FULL_COL     = {_SS_FULL_COL!r}')
print()
print('Primeras 3 filas:')
_df_ss.head(3)
print(f'  _SS_FULL_COL     = {_SS_FULL_COL!r}')
print()
print('Primeras 3 filas:')
_df_ss.head(3)


Columnas SS:
['Researcher_id', 'First_year', 'Last_year', 'Ranking_position_citations', 'Citations', 'Productivity', 'Collaborations', 'Field', 'Gender', 'Name', 'Probability', 'Clean_standarized_name', 'Career_age', 'Number_of_genders', 'Gender_Namsor', 'Gender_probability_Namsor', 'Combined_gender']

Auto-detect:


NameError: name '_SS_NAME_COL' is not defined

## Manual review step

Open `manual_validation_20.csv` in Excel/LibreOffice and fill in the empty columns for each persona row:

- `found_in_ss_manual`: `yes` / `no` / empty (unknown). Did you find this exact researcher on Semantic Scholar?
- `found_in_oa_manual`: same, for OpenAlex.
- `affiliation_correct_manual`: `yes` / `no` / empty. Does at least one institution in `current_affiliations` match a real affiliation of this researcher?
- `field_correct_manual`: `yes` / `no` / empty. Does the researcher actually work in the requested `field`?
- `notes_manual`: free text — record anything notable (e.g. "same name but different person", "affiliation outdated").

When done, re-run the cell below to compute concordance.

In [4]:
# 5. Concordance after manual filling. Re-run after editing the CSV.
df_filled = pd.read_csv('/home/asanchez/code/asanchez/LLMScholar-Personas/results/manual_v2/manual_validation_20_v2.csv', sep=',', dtype=str).fillna('')

# ── Enrich con oa_status desde factuality_full.csv (no requiere regenerar CSV) ──
# Si el CSV manual no tiene `oa_status_auto`, lo traemos del pipeline join-on
# (model, language, role, task, location, k, target, field, subfield, run_id, name, lastname).
if 'oa_status_auto' not in df_filled.columns:
    join_keys = ['model','language','role','task','location','k','target','field','subfield','run_id','name','lastname']
    print(f'Trayendo oa_status desde factuality_full …')
    df_pipe = pd.read_csv(FACT_FULL_CSV, low_memory=False,
                          usecols=lambda c: c in join_keys + ['oa_status'])
    df_pipe = df_pipe.dropna(subset=join_keys).drop_duplicates(subset=join_keys)
    # Casteamos las keys a str para joinear con df_filled (que viene dtype=str)
    for k in join_keys:
        df_pipe[k] = df_pipe[k].astype(str)
    df_filled = df_filled.merge(df_pipe.rename(columns={'oa_status': 'oa_status_auto'}),
                                on=join_keys, how='left')
    df_filled['oa_status_auto'] = df_filled['oa_status_auto'].fillna('')
    n_with_oa = (df_filled['oa_status_auto'] != '').sum()
    print(f'  oa_status_auto rellenado en {n_with_oa}/{len(df_filled)} filas')


def truthy(v):
    return str(v).strip().lower() in ('yes','y','true','1','t')

def manual_found_any_row(r):
    """True/False/None: True si en SS o OA encontraste; False si pusiste 'no' en ambos; None si vacío."""
    ss = r['found_in_ss_manual']
    oa = r['found_in_oa_manual']
    if not ss and not oa:
        return None
    return truthy(ss) or truthy(oa)

def auto_found_row(r):
    """auto_found = found en SS O found en OA (simétrico al criterio manual)."""
    ss = r.get('author_status_auto', '')
    oa = r.get('oa_status_auto', '')
    if not ss and not oa:
        return None
    return (ss == 'found') or (oa == 'found')

df_filled['manual_found_any'] = df_filled.apply(manual_found_any_row, axis=1)
df_filled['auto_found']       = df_filled.apply(auto_found_row,       axis=1)

# ── Author concordance (auto = SS OR OA) ──────────────────────────────────────
both_labeled = df_filled[df_filled['manual_found_any'].notna() & df_filled['auto_found'].notna()]
n_compared = len(both_labeled)
n_agree    = (both_labeled['manual_found_any'] == both_labeled['auto_found']).sum()
if n_compared > 0:
    print(f'\nAUTHOR FOUND concordance (auto = SS OR OA): {n_agree}/{n_compared} = {100*n_agree/n_compared:.1f}%')
    print()
    print('Confusion (auto_found × manual_found_any):')
    print(both_labeled.groupby(['auto_found','manual_found_any']).size().unstack(fill_value=0))
else:
    print('No manual labels filled yet — fill the CSV then re-run.')

# Per-request breakdown
if n_compared > 0:
    by_req = both_labeled.groupby('request_id').apply(
        lambda g: pd.Series({
            'n_personas': len(g),
            'agree': (g['manual_found_any'] == g['auto_found']).sum(),
        }),
        include_groups=False,
    )
    by_req['pct'] = (100 * by_req['agree'] / by_req['n_personas']).round(1)
    print()
    print('Por request:')
    print(by_req)

# ── Affiliation concordance — versión honesta ─────────────────────────────────
if 'affiliation_status_auto' in df_filled.columns:
    real_found = df_filled[
        (df_filled['auto_found']        == True) &
        (df_filled['manual_found_any']  == True) &
        (df_filled['affiliation_correct_manual'].str.strip() != '') &
        (df_filled['affiliation_status_auto'].str.strip()     != '')
    ].copy()

    if len(real_found) > 0:
        real_found['auto_aff_match']   = real_found['affiliation_status_auto'] == 'affiliation_match'
        real_found['manual_aff_match'] = real_found['affiliation_correct_manual'].apply(truthy)
        n_aff   = len(real_found)
        ag_aff  = (real_found['auto_aff_match'] == real_found['manual_aff_match']).sum()
        print()
        print(f'AFFILIATION concordance (solo autores realmente encontrados): {ag_aff}/{n_aff} = {100*ag_aff/n_aff:.1f}%')
        print()
        print('Confusion (auto_aff_match × manual_aff_match):')
        print(real_found.groupby(['auto_aff_match','manual_aff_match']).size().unstack(fill_value=0))
    else:
        print()
        print('AFFILIATION concordance: no hay autores realmente encontrados con afiliación etiquetada.')

# ── Field concordance — misma lógica honesta ──────────────────────────────────
if 'field_status_auto' in df_filled.columns and (df_filled['field_correct_manual'].str.strip() != '').any():
    real_found_field = df_filled[
        (df_filled['auto_found']        == True) &
        (df_filled['manual_found_any']  == True) &
        (df_filled['field_correct_manual'].str.strip()  != '') &
        (df_filled['field_status_auto'].str.strip()      != '')
    ].copy()
    if len(real_found_field) > 0:
        real_found_field['auto_field_match']   = real_found_field['field_status_auto'] == 'field_match'
        real_found_field['manual_field_match'] = real_found_field['field_correct_manual'].apply(truthy)
        n_f  = len(real_found_field)
        ag_f = (real_found_field['auto_field_match'] == real_found_field['manual_field_match']).sum()
        print()
        print(f'FIELD concordance (solo autores realmente encontrados): {ag_f}/{n_f} = {100*ag_f/n_f:.1f}%')



AUTHOR FOUND concordance (auto = SS OR OA): 35/37 = 94.6%

Confusion (auto_found × manual_found_any):
manual_found_any  False  True 
auto_found                    
False                15      1
True                  1     20

Por request:
            n_personas  agree    pct
request_id                          
0                    1      1  100.0
1                    1      1  100.0
11                   5      5  100.0
12                   5      4   80.0
13                   5      5  100.0
14                   4      4  100.0
15                   1      1  100.0
16                   1      1  100.0
19                   1      1  100.0
2                    5      5  100.0
3                    5      4   80.0
4                    1      1  100.0
6                    1      1  100.0
8                    1      1  100.0

AFFILIATION concordance (solo autores realmente encontrados): 18/20 = 90.0%

Confusion (auto_aff_match × manual_aff_match):
manual_aff_match  False  True 
auto_aff_ma

In [ ]:
from pathlib import Path
csv_path = Path("/home/asanchez/code/asanchez/LLMScholar-Personas/data/annotator_agreement/manual_validation_20_checked.csv")
lines = csv_path.read_text().splitlines()
print(f'Total líneas: {len(lines)}')
print(f'Header ({lines[0].count(",")+1} campos):')
print(lines[0])
print(f'\nLínea 21 ({lines[20].count(",")+1} campos):')
print(lines[20])
print(f'\nLínea 22:')
print(lines[21])

Total líneas: 39
Header (1 campos):
request_id;json_file;json_key;model;language;role;task;location;k;target;field;subfield;run_id;name;lastname;current_affiliations;areas_of_research_or_work;reason;source;author_status_auto;field_status_auto;seniority_status_auto;location_status_auto;affiliation_status_auto;affiliation_best_match_score;affiliation_best_match_oa;affiliation_oa_all;ss_search_url;oa_search_url;found_in_ss_manual;found_in_oa_manual;affiliation_correct_manual;field_correct_manual;notes_manual

Línea 21 (15 campos):
11;ollama_english_mistral-nemo-12b-instruct-2407-q4_K_M.json;476;mistral-nemo:12b-instruct-2407-q4_K_M;english;PhD student;seeking an advisor;Germany;5;Junior Professor;Psychology;Forensic Psychology;2;Jan;Hoffmann;[{'position': 'Junior Professor', 'affiliation': 'Institute of Psychology, Humboldt-Universität zu Berlin;['Forensic Psychology', 'Criminal Cognition', 'False Confessions'];He has a proven track record of collaborative research and has published in to

In [5]:
excluded = df_filled[
      df_filled['manual_found_any'].isna() | df_filled['auto_found'].isna()
  ]
print(f'{len(excluded)} filas excluidas:\n')
print(excluded[['request_id','name','lastname',
                'found_in_ss_manual','found_in_oa_manual',
                'author_status_auto']].to_string())

1 filas excluidas:

   request_id   name lastname found_in_ss_manual found_in_oa_manual author_status_auto
32         14  Erika    Muñoz                 no                 no                   


## Precision resumen

Una sola tabla con la **precision** del pipeline automático contra las
etiquetas manuales: de todas las personas que el algoritmo marcó como
`found` (o `field_match` / `affiliation_match`), ¿cuántas confirmé yo en
la revisión manual?

Precision = manual_yes / auto_yes


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Precision resumen — de los que el pipeline marcó como found/match,
# ¿cuántos confirmé manualmente?
# ─────────────────────────────────────────────────────────────────────────────
def truthy(v):
    return str(v).strip().lower() in ('yes', 'y', 'true', '1', 't')

def has_text(s):
    return s.fillna('').astype(str).str.strip() != ''

rows = []

# 1) AUTHOR FOUND — precision: de los que auto marcó 'found', cuántos confirmé manual
#    auto_found = (author_status='found' OR oa_status='found')
#    manual     = (found_in_ss='yes' OR found_in_oa='yes')
mask = (df_filled['auto_found'] == True) & df_filled['manual_found_any'].notna()
df_a = df_filled[mask]
if len(df_a) > 0:
    correct = int(df_a['manual_found_any'].sum())
    rows.append({'dimension': 'author_found',
                 'n_auto_found': len(df_a),
                 'n_manual_confirmed': correct,
                 'precision_%': round(100 * correct / len(df_a), 1)})

# 2) AUTHOR FOUND (SS-only) — precision de SS
df_ss = df_filled[(df_filled['author_status_auto'] == 'found') &
                   has_text(df_filled['found_in_ss_manual'])].copy()
if len(df_ss) > 0:
    df_ss['manual'] = df_ss['found_in_ss_manual'].apply(truthy)
    correct = int(df_ss['manual'].sum())
    rows.append({'dimension': 'author_found_ss',
                 'n_auto_found': len(df_ss),
                 'n_manual_confirmed': correct,
                 'precision_%': round(100 * correct / len(df_ss), 1)})

# 3) AUTHOR FOUND (OA-only) — precision de OA
df_oa = df_filled[(df_filled['oa_status_auto'] == 'found') &
                   has_text(df_filled['found_in_oa_manual'])].copy()
if len(df_oa) > 0:
    df_oa['manual'] = df_oa['found_in_oa_manual'].apply(truthy)
    correct = int(df_oa['manual'].sum())
    rows.append({'dimension': 'author_found_oa',
                 'n_auto_found': len(df_oa),
                 'n_manual_confirmed': correct,
                 'precision_%': round(100 * correct / len(df_oa), 1)})

# 4) FIELD — de los que auto marcó 'field_match', cuántos confirmé manual
if 'field_status_auto' in df_filled.columns:
    df_f = df_filled[(df_filled['field_status_auto'] == 'field_match') &
                      has_text(df_filled['field_correct_manual'])].copy()
    if len(df_f) > 0:
        df_f['manual'] = df_f['field_correct_manual'].apply(truthy)
        correct = int(df_f['manual'].sum())
        rows.append({'dimension': 'field_match',
                     'n_auto_found': len(df_f),
                     'n_manual_confirmed': correct,
                     'precision_%': round(100 * correct / len(df_f), 1)})

# 5) AFFILIATION — de los que auto marcó 'affiliation_match', cuántos confirmé manual
if 'affiliation_status_auto' in df_filled.columns:
    df_af = df_filled[(df_filled['affiliation_status_auto'] == 'affiliation_match') &
                       has_text(df_filled['affiliation_correct_manual'])].copy()
    if len(df_af) > 0:
        df_af['manual'] = df_af['affiliation_correct_manual'].apply(truthy)
        correct = int(df_af['manual'].sum())
        rows.append({'dimension': 'affiliation_match',
                     'n_auto_found': len(df_af),
                     'n_manual_confirmed': correct,
                     'precision_%': round(100 * correct / len(df_af), 1)})

COLS = ['dimension', 'n_auto_found', 'n_manual_confirmed', 'precision_%']
accuracy_df = pd.DataFrame(rows, columns=COLS)
if len(accuracy_df) == 0:
    print('No hay filas que cumplan los filtros (¿están las columnas manuales llenas?).')
else:
    accuracy_df = accuracy_df.set_index('dimension')
print('Precision del pipeline automático (de los marcados como found/match, ¿cuántos confirmé yo?):')
print()
accuracy_df


Precision del pipeline automático (de los marcados como found/match, ¿cuántos confirmé yo?):



,n_auto_found,n_manual_confirmed,precision_%
dimension,,,
author_found,21,20,95.2
author_found_ss,11,10,90.9
author_found_oa,20,19,95.0
field_match,2,2,100.0
affiliation_match,1,1,100.0


---

## Muestra alternativa: 10 recomendaciones random

Sampling directo sobre `factuality_full.csv` (1 row = 1 recomendación, no agrupado por request). Más simple que el flujo de 20-requests: no levanta JSONs ni reconstruye personas — usa lo que el pipeline ya guardó. Usa el `summary_v2/factuality_full.csv` cargado más arriba.

Poné `RANDOM_STATE_RECS = None` para muestra distinta cada corrida; con un int queda reproducible.

In [7]:
# ── Generar manual_validation_10_random.csv ─────────────────────────────────
from urllib.parse import quote

OUT_RANDOM = OUT_DIR / 'manual_validation_10_random_v2.csv'

N_RECS             = 10
RANDOM_STATE_RECS  = 42        # None para muestra distinta cada corrida

KEEP_COLS = [
    'model','language','role','task','location','k','target','field','subfield','run_id',
    'name','lastname','current_affiliations','areas_of_research_or_work','reason','source',
    'author_status','oa_status','field_status','seniority_status','location_status',
    'affiliation_status','affiliation_best_match_score','affiliation_best_match_oa','affiliation_oa_all',
    'matched_name','researcher_id','match_score',
    'oa_id','oa_display_name','oa_match_score',
    'gt_field','gt_citations',
]
df_v2 = pd.read_csv(FACT_FULL_CSV, low_memory=False,
                    usecols=lambda c: c in KEEP_COLS)
print(f'factuality_full v2 rows: {len(df_v2):,}')

sample = df_v2.sample(n=N_RECS, random_state=RANDOM_STATE_RECS).reset_index(drop=True)

# Renombrar a *_auto para alinearlo con el CSV de 20-requests
rename_auto = {
    'author_status':      'author_status_auto',
    'oa_status':          'oa_status_auto',
    'field_status':       'field_status_auto',
    'seniority_status':   'seniority_status_auto',
    'location_status':    'location_status_auto',
    'affiliation_status': 'affiliation_status_auto',
}
sample = sample.rename(columns=rename_auto)

# URLs de búsqueda para el revisor humano
def _q(n, l): return quote(f'{n} {l}'.strip())
sample['ss_search_url'] = sample.apply(lambda r: f'https://www.semanticscholar.org/search?q={_q(r["name"], r["lastname"])}', axis=1)
sample['oa_search_url'] = sample.apply(lambda r: f'https://api.openalex.org/authors?search={_q(r["name"], r["lastname"])}', axis=1)

# Columnas manuales vacías
for c in ('found_in_ss_manual','found_in_oa_manual','affiliation_correct_manual','field_correct_manual','notes_manual'):
    sample[c] = ''

# Orden final
ordered = (
    ['model','language','role','task','location','k','target','field','subfield','run_id',
     'name','lastname','current_affiliations','areas_of_research_or_work','reason','source',
     'author_status_auto','oa_status_auto','field_status_auto','seniority_status_auto','location_status_auto',
     'affiliation_status_auto','affiliation_best_match_score','affiliation_best_match_oa','affiliation_oa_all',
     'matched_name','researcher_id','match_score','gt_field','gt_citations',
     'oa_id','oa_display_name','oa_match_score',
     'ss_search_url','oa_search_url',
     'found_in_ss_manual','found_in_oa_manual','affiliation_correct_manual','field_correct_manual','notes_manual']
)
sample = sample[[c for c in ordered if c in sample.columns]]

sample.to_csv(OUT_RANDOM, index=False)
print(f'Wrote {len(sample)} recomendaciones random → {OUT_RANDOM}')
sample[['name','lastname','model','language','field','author_status_auto','oa_status_auto','field_status_auto','affiliation_status_auto']]

factuality_full v2 rows: 3,907,448
Wrote 10 recomendaciones random → /data/datasets/LLMScholar-Personas/results/manual_v2/manual_validation_10_random_v2.csv


,name,lastname,model,language,field,author_status_auto,oa_status_auto,field_status_auto,affiliation_status_auto
0,Masato Nakazawa,Nakazawa,yi:9b-chat-v1.5-q4_K_M,spanish,Biología,found,not_found,field_match,affiliation_unknown
1,Bob White,Davis,mistral:7b-instruct-v0.3-q4_K_M,german,Mathematik,hallucinated,not_found,not_applicable,not_applicable
2,Sho,Nakamura,gpt-4.1-mini-2025-04-14,german,Physik,hallucinated,found,not_applicable,not_applicable
3,Dolphin,Assistant,dolphin-mixtral:8x22b-v2.9-q4_K_M,spanish,Física,hallucinated,not_found,not_applicable,not_applicable
4,Suzanne,Little,llama3.3:70b-instruct-q4_K_M,german,Informatik,hallucinated,found,not_applicable,not_applicable
5,Ingrid,Koch,phi4-reasoning:14b-q4_K_M,spanish,Sociología,hallucinated,found,not_applicable,not_applicable
6,Inga,Kryshtal,gpt-4.1-2025-04-14,english,Mathematics,hallucinated,not_found,not_applicable,not_applicable
7,Dr. Tshegofatso L. Phala,Phala,dolphin3:8b-llama3.1-q4_K_M,german,Informatik,hallucinated,not_found,not_applicable,not_applicable
8,Felix,Bauer,mixtral:8x7b-instruct-v0.1-q4_K_M,spanish,Psicología,hallucinated,found,not_applicable,not_applicable
9,Hiroshi,Katori,gpt-4.1-mini-2025-04-14,spanish,Física,hallucinated,not_found,not_applicable,not_applicable
